In [ ]:
# secp256k1 parameters — the numbers that secure Bitcoin

# Field prime: coordinates live in F_P
SECP_P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F

# Group order: scalars (private keys) live in Z_N  
SECP_N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

# Generator point G: the "starting point" for all key generation
SECP_GX = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
SECP_GY = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8

# Curve coefficients
A_COEFF = 0
B_COEFF = 7

print("=== secp256k1 Parameters ===")
print(f"Curve: y² = x³ + {A_COEFF}x + {B_COEFF}")
print(f"P = 2²⁵⁶ - 2³² - 977")
print(f"  = {SECP_P}")
print(f"  ({SECP_P.bit_length()} bits)")
print(f"\nN = {SECP_N}")
print(f"  ({SECP_N.bit_length()} bits)")
print(f"\nNon-singular check: 4(0)³ + 27(7)² = {4*0**3 + 27*7**2} ≠ 0  ✓")
print(f"\nP mod 4 = {SECP_P % 4}  (enables efficient square roots)")

from typing import Optional

class Point:
    """A point on secp256k1 (or infinity)."""
    def __init__(self, x: Optional[int] = None, y: Optional[int] = None):
        self.x = x
        self.y = y
    
    def is_infinity(self) -> bool:
        return self.x is None or self.y is None
    
    def copy(self) -> 'Point':
        return Point(self.x, self.y)
    
    def __eq__(self, other):
        if self.is_infinity() and other.is_infinity():
            return True
        return self.x == other.x and self.y == other.y
    
    def __repr__(self):
        if self.is_infinity():
            return "O (point at infinity)"
        return f"({hex(self.x)[:16]}..., {hex(self.y)[:16]}...)"

G = Point(SECP_GX, SECP_GY)
INFINITY = Point()  # The identity element

print(f"Generator G:     {G}")
print(f"Identity O:      {INFINITY}")

def mod_inverse(a: int, p: int) -> int:
    """a⁻¹ mod p via Fermat's little theorem: a^(p-2) mod p."""
    return pow(a, p - 2, p)

def mod_sqrt(a: int, p: int) -> int:
    """√a mod p for p ≡ 3 (mod 4): a^((p+1)/4) mod p."""
    return pow(a, (p + 1) // 4, p)

def point_add(p1: Point, p2: Point) -> Point:
    """
    Add two distinct points on secp256k1.
    
    Slope:  λ = (y₂ - y₁) / (x₂ - x₁) mod P
    Result: x₃ = λ² - x₁ - x₂
            y₃ = λ(x₁ - x₃) - y₁
    """
    if p1.is_infinity():
        return p2.copy()
    if p2.is_infinity():
        return p1.copy()
    
    if p1.x == p2.x:
        if (p1.y + p2.y) % SECP_P == 0:
            return Point()  # P + (-P) = O
        return point_double(p1)
    
    lam = ((p2.y - p1.y) * mod_inverse(p2.x - p1.x, SECP_P)) % SECP_P
    x3 = (lam * lam - p1.x - p2.x) % SECP_P
    y3 = (lam * (p1.x - x3) - p1.y) % SECP_P
    return Point(x3, y3)

def point_double(p: Point) -> Point:
    """
    Double a point on secp256k1.
    
    Slope:  λ = 3x² / 2y mod P  (tangent to curve at P)
    Result: x₃ = λ² - 2x
            y₃ = λ(x - x₃) - y
    """
    if p.is_infinity() or p.y == 0:
        return Point()
    
    lam = (3 * p.x * p.x * mod_inverse(2 * p.y, SECP_P)) % SECP_P
    x3 = (lam * lam - 2 * p.x) % SECP_P
    y3 = (lam * (p.x - x3) - p.y) % SECP_P
    return Point(x3, y3)

def point_negate(p: Point) -> Point:
    """Negate: -P = (x, -y mod P)."""
    if p.is_infinity():
        return Point()
    return Point(p.x, (SECP_P - p.y) % SECP_P)

# Verify G is on the curve
lhs = (G.y * G.y) % SECP_P
rhs = (G.x ** 3 + 7) % SECP_P
print(f"G on curve? y² mod P == x³+7 mod P: {lhs == rhs}  ✓")

# Verify group properties
G2 = point_double(G)
print(f"\n2G = {G2}")
print(f"G + O = G? {point_add(G, INFINITY) == G}  ✓  (identity)")
neg_G = point_negate(G)
print(f"G + (-G) = O? {point_add(G, neg_G).is_infinity()}  ✓  (inverse)")

def scalar_mult(k: int, p: Point) -> Point:
    """Compute k × P using double-and-add. O(log k) operations."""
    if k == 0 or p.is_infinity():
        return Point()
    k = k % SECP_N
    if k == 0:
        return Point()
    
    result = Point()  # Start at O (identity)
    addend = p.copy()
    
    while k > 0:
        if k & 1:
            result = point_add(result, addend)
        addend = point_double(addend)
        k >>= 1
    
    return result

# Verify: 3G computed two ways
G3_algo = scalar_mult(3, G)
G3_manual = point_add(G, point_double(G))
print(f"3G (double-and-add): {G3_algo}")
print(f"3G (G + 2G):         {G3_manual}")
print(f"Match: {G3_algo == G3_manual}  ✓")

# The fundamental property: N × G = O (wraps around)
# (Don't actually compute this — it would take forever)
# But we can verify: (N-1)×G + G = O
print(f"\nFundamental: N × G = O (point at infinity)")
print(f"This means private key space is cyclic with order N.")
print(f"N ≈ 1.16 × 10⁷⁷ — more than atoms in the observable universe.")

def serialize_compressed(p: Point) -> bytes:
    """Point → 33-byte compressed public key."""
    prefix = 0x03 if (p.y & 1) else 0x02
    return bytes([prefix]) + p.x.to_bytes(32, 'big')

def parse_compressed(data: bytes) -> Point:
    """33-byte compressed public key → Point."""
    x = int.from_bytes(data[1:], 'big')
    y2 = (pow(x, 3, SECP_P) + 7) % SECP_P
    y = mod_sqrt(y2, SECP_P)
    if (y & 1) != (data[0] == 0x03):
        y = SECP_P - y
    return Point(x, y)

# Demo: generate a key pair
import secrets
private_key = secrets.randbelow(SECP_N - 1) + 1
public_key = scalar_mult(private_key, G)
compressed = serialize_compressed(public_key)

print(f"=== Key Pair Generation ===")
print(f"Private key (d):  {hex(private_key)[:20]}...")
print(f"Public key (P = d×G):")
print(f"  x: {hex(public_key.x)}")
print(f"  y: {hex(public_key.y)}")
print(f"Compressed: {compressed.hex()}")
print(f"  Prefix 0x{compressed[0]:02x} → y is {'odd' if compressed[0] == 0x03 else 'even'}")

# Round-trip verification
recovered = parse_compressed(compressed)
print(f"\nRound-trip: {public_key == recovered}  ✓")


In [18]:
# Side-by-side: ElGamal vs ECC key exchange

print("=" * 60)
print("ElGamal Key Exchange (integers mod p)")
print("=" * 60)

# Small ElGamal example (from the essay)
p_eg = 101
alpha = 7  # Primitive root mod 101

# Alice
a_priv = 23  # Alice's private key
beta_a = pow(alpha, a_priv, p_eg)  # Alice's public key
print(f"Alice: private a={a_priv}, public β = α^a mod p = 7^{a_priv} mod 101 = {beta_a}")

# Bob
b_priv = 37  # Bob's private key
beta_b = pow(alpha, b_priv, p_eg)  # Bob's public key
print(f"Bob:   private b={b_priv}, public β'= α^b mod p = 7^{b_priv} mod 101 = {beta_b}")

# Shared secret
shared_eg_a = pow(beta_b, a_priv, p_eg)  # Alice computes β'^a
shared_eg_b = pow(beta_a, b_priv, p_eg)  # Bob computes β^b
print(f"Shared secret: Alice={shared_eg_a}, Bob={shared_eg_b}, Match={shared_eg_a == shared_eg_b}")
print(f"(Both compute α^(ab) mod p = 7^{a_priv*b_priv} mod 101 = {pow(alpha, a_priv*b_priv, p_eg)})")

print(f"\n{'=' * 60}")
print("ECC Key Exchange (ECDH on secp256k1)")
print("=" * 60)

# Alice
alice_priv = secrets.randbelow(SECP_N - 1) + 1
alice_pub = scalar_mult(alice_priv, G)  # Alice's public key = a × G
print(f"Alice: private a (256-bit random), public A = a×G")

# Bob
bob_priv = secrets.randbelow(SECP_N - 1) + 1
bob_pub = scalar_mult(bob_priv, G)  # Bob's public key = b × G
print(f"Bob:   private b (256-bit random), public B = b×G")

# Shared secret: both compute the same point
shared_ecc_a = scalar_mult(alice_priv, bob_pub)   # a × (b×G) = ab×G
shared_ecc_b = scalar_mult(bob_priv, alice_pub)    # b × (a×G) = ab×G
print(f"Shared secret point: {shared_ecc_a == shared_ecc_b}  ✓")
print(f"Both compute a×b×G (same point, never transmitted)")

print(f"\n{'─' * 60}")
print(f"ElGamal: security from α^a mod p  (needs ~3072-bit p)")
print(f"ECC:     security from k×G         (needs ~256-bit N)")
print(f"Same security, 12× smaller keys.")

ElGamal Key Exchange (integers mod p)
Alice: private a=23, public β = α^a mod p = 7^23 mod 101 = 27
Bob:   private b=37, public β'= α^b mod p = 7^37 mod 101 = 35
Shared secret: Alice=94, Bob=94, Match=True
(Both compute α^(ab) mod p = 7^851 mod 101 = 94)

ECC Key Exchange (ECDH on secp256k1)
Alice: private a (256-bit random), public A = a×G
Bob:   private b (256-bit random), public B = b×G
Shared secret point: True  ✓
Both compute a×b×G (same point, never transmitted)

────────────────────────────────────────────────────────────
ElGamal: security from α^a mod p  (needs ~3072-bit p)
ECC:     security from k×G         (needs ~256-bit N)
Same security, 12× smaller keys.


## 4.2 ECC Encryption / Decryption (ElGamal on Curves)

The essay demonstrates ECC encryption using the curve $y^2 = x^3 - x + 4$ over $\mathbb{F}_{457}$.
The scheme maps directly from ElGamal:

| ElGamal | ECC Analog |
|---------|------------|
| $\beta = \alpha^a \bmod p$ | $Q = d \times G$ (public key) |
| Encrypt: $(\alpha^k, m \cdot \beta^k)$ | Encrypt: $(k \times G, \; P_m + k \times Q)$ |
| Decrypt: $m = t \cdot (\beta')^{-a}$ | Decrypt: $P_m = C_2 - d \times C_1$ |

Multiplication in ElGamal becomes **point addition** in ECC.  
Exponentiation becomes **scalar multiplication**.

In [19]:
# ECC encryption/decryption on a small curve (from the essay)
# Curve: y² = x³ - x + 4 over F_457

P_SMALL = 457
A_SMALL = -1  # a coefficient
B_SMALL = 4   # b coefficient

class SmallPoint:
    def __init__(self, x=None, y=None):
        self.x = x
        self.y = y
    def is_infinity(self):
        return self.x is None
    def copy(self):
        return SmallPoint(self.x, self.y)
    def __eq__(self, other):
        if self.is_infinity() and other.is_infinity(): return True
        return self.x == other.x and self.y == other.y
    def __repr__(self):
        if self.is_infinity(): return "O"
        return f"({self.x}, {self.y})"

def small_add(p1, p2):
    if p1.is_infinity(): return p2.copy()
    if p2.is_infinity(): return p1.copy()
    if p1.x == p2.x:
        if (p1.y + p2.y) % P_SMALL == 0:
            return SmallPoint()
        lam = ((3 * p1.x * p1.x + A_SMALL) * pow(2 * p1.y, P_SMALL - 2, P_SMALL)) % P_SMALL
    else:
        lam = ((p2.y - p1.y) * pow(p2.x - p1.x, P_SMALL - 2, P_SMALL)) % P_SMALL
    x3 = (lam * lam - p1.x - p2.x) % P_SMALL
    y3 = (lam * (p1.x - x3) - p1.y) % P_SMALL
    return SmallPoint(x3, y3)

def small_mult(k, p):
    if k == 0 or p.is_infinity(): return SmallPoint()
    result = SmallPoint()
    addend = p.copy()
    while k > 0:
        if k & 1:
            result = small_add(result, addend)
        addend = small_add(addend, addend)
        k >>= 1
    return result

def small_negate(p):
    if p.is_infinity(): return SmallPoint()
    return SmallPoint(p.x, (P_SMALL - p.y) % P_SMALL)

def small_sqrt(a, p):
    """Square root mod p using Tonelli-Shanks (works for any odd prime)."""
    if a % p == 0:
        return 0
    if pow(a, (p - 1) // 2, p) != 1:
        return None
    if p % 4 == 3:
        return pow(a, (p + 1) // 4, p)
    q, s = p - 1, 0
    while q % 2 == 0:
        q //= 2
        s += 1
    z = 2
    while pow(z, (p - 1) // 2, p) != p - 1:
        z += 1
    m, c, t, r = s, pow(z, q, p), pow(a, q, p), pow(a, (q + 1) // 2, p)
    while t != 1:
        i = 1
        tmp = (t * t) % p
        while tmp != 1:
            tmp = (tmp * tmp) % p
            i += 1
        b = pow(c, 1 << (m - i - 1), p)
        m, c, t, r = i, (b * b) % p, (t * b * b) % p, (r * b) % p
    return r

# Setup: (4, 8) is on y² = x³ - x + 4 over F_457 since 8²=64 and 4³-4+4=64
G_small = SmallPoint(4, 8)
assert (8**2) % P_SMALL == (4**3 + A_SMALL*4 + B_SMALL) % P_SMALL, "G not on curve!"

# Key generation
d = 101  # Private key (small for demo)
Q = small_mult(d, G_small)  # Public key
print(f"=== ECC Encryption (Essay Example) ===")
print(f"Curve: y² = x³ - x + 4 over F_457")
print(f"G = {G_small}")
print(f"Private key d = {d}")
print(f"Public key Q = d×G = {Q}")

# Encoding: map message character to point (k=30 from essay)
K_ENC = 30

def encode_char_to_point(m_val):
    """Koblitz encoding: find point with x near K_ENC*m."""
    for j in range(K_ENC):
        x = K_ENC * m_val + j
        rhs = (x**3 + A_SMALL * x + B_SMALL) % P_SMALL
        y = small_sqrt(rhs, P_SMALL)
        if y is not None:
            return SmallPoint(x, y)
    return None

def decode_point_to_char(pt):
    return pt.x // K_ENC

# Encrypt 'H' (value = 7 in A=0..Z=25)
msg_val = 7  # 'H'
pm = encode_char_to_point(msg_val)
print(f"\nMessage: 'H' (value {msg_val}) → point {pm}")

# Encryption: (k×G, Pm + k×Q) with random k
k_rand = 41
C1 = small_mult(k_rand, G_small)       # k × G
C2 = small_add(pm, small_mult(k_rand, Q))  # Pm + k×Q
print(f"\nEncrypt with random k={k_rand}:")
print(f"  C1 = k×G = {C1}")
print(f"  C2 = Pm + k×Q = {C2}")

# Decryption: Pm = C2 - d×C1
dC1 = small_mult(d, C1)  # d × C1 = d×k×G = k×Q
pm_recovered = small_add(C2, small_negate(dC1))  # C2 - d×C1
msg_recovered = decode_point_to_char(pm_recovered)
print(f"\nDecrypt:")
print(f"  d×C1 = {dC1}")
print(f"  Pm = C2 - d×C1 = {pm_recovered}")
print(f"  Decoded: value {msg_recovered} → '{chr(65 + msg_recovered)}'")
print(f"  Match: {pm == pm_recovered}  ✓")

=== ECC Encryption (Essay Example) ===
Curve: y² = x³ - x + 4 over F_457
G = (4, 8)
Private key d = 101
Public key Q = d×G = (81, 387)

Message: 'H' (value 7) → point (210, 298)

Encrypt with random k=41:
  C1 = k×G = (445, 49)
  C2 = Pm + k×Q = (11, 180)

Decrypt:
  d×C1 = (427, 190)
  Pm = C2 - d×C1 = (210, 298)
  Decoded: value 7 → 'H'
  Match: True  ✓


### Why decryption works

$$C_2 - d \times C_1 = (P_m + k \times Q) - d \times (k \times G)$$
$$= P_m + k \times (d \times G) - d \times (k \times G)$$
$$= P_m + k \cdot d \times G - d \cdot k \times G$$
$$= P_m \quad \checkmark$$

The random factor $k$ cancels out because both parties have access to the
shared secret $k \cdot d \times G$ through different paths.

---